# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [6]:
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display
from openai import OpenAI

In [12]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

# Ollama OpenAI-compatible API (run `ollama serve`; default port 11434)
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [8]:
load_dotenv()
openai = OpenAI()

In [ ]:
system_prompt = """
You are a helpful technical tutor who answers questions about code, 
software engineering, data science and LLMs. 
Respond in markdown format.
"""

def answer_question(openai_client, question, model=MODEL_GPT, stream=False):
    if stream:
        stream_response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question}
            ],
            stream=True,
        )
        markdown_text = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream_response:
            markdown_text += chunk.choices[0].delta.content or ""
            update_display(Markdown(markdown_text), display_id=display_handle.display_id)
        return markdown_text
    else:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": question}
            ],
        )
        return response.choices[0].message.content

In [10]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why
language: python
code:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
result_gpt = answer_question(openai, question, MODEL_GPT, stream=True)

In [13]:
openai_llama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
result_llama = answer_question(openai_llama, question, MODEL_LLAMA)

In [14]:
display(Markdown(result_llama))

**Code Explanation**
=====================

The given Python code uses a feature called **context managers** with the `yield from` keyword. Here's what it does:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Let's break it down:

*   `{book.get("author") for book in books}` is a **set comprehension**, which creates an iterable set of values.
    *   The outermost `for` loop iterates over the elements in the `books` collection, presumably a list or dictionary of books (e.g., a JSON array or a pandas DataFrame).
    *   Within this loop, each book is processed to extract its associated author information, using the `get()` method. If the author key doesn't exist for a particular book, the `get()` method returns `None` (in case you're not handling these cases in your code).
*   The `yield from` keyword tells Python that this iterable expression should be treated as a **context manager**.
    *   This syntax allows us to simplify the creation of context managers by "yielding" the items one at a time, instead of having to return an entire list.
    *   In fact, when you use `yield from`, you don't need to repeat logic that's already present elsewhere in your code (like creating an empty list and appending results).

**Why is this useful?**
------------------------

This specific use case might not seem directly applicable in most scenarios, but context managers can be incredibly powerful tools for certain problems. Some common examples include:

*   **Handling streams**: Context managers are perfect when dealing with large datasets that don't fit neatly into memory (e.g., CSV readers or database cursors). You can process these large datasets one entry at a time using a yield.
*   **Working with nested lists or dictionaries**: When working with larger structures, such as nested lists or dictionaries, context managers help manage iterations for you.

This particular code snippet has some drawbacks:
*   The set comprehension isn't actually used for deduplication or set operations since there's no reason to iterate over the same data again.
*   Creating a dictionary of books (e.g., with author keys in JSON) and then applying this code won't result in a very efficient structure.

To see if you have use cases where yield-from can be utilized to ease development, here is an extended example utilizing context manager to load `books` into memory from your data stream: 
###  A more suitable context manager
```python
import json

def book_iterator(books):
    for book in books:
        # handle case when the author information isn't present for the given book,
        # returning a value that can easily be detected, if necessary
        yield (
            book.get("author")
            or "Untitled"
            or json.dumps(book)
        )

data = [
    {"id": 1, "title": "Book One"},
    {"id":2, "title":"Second Book"}
]

books = [book for book in data]
books_with Authors =  {book[0]: book for book in books}

for author in book_iterator(books):
   print(author)
```
**Final Evaluation**
--------------------

Keep the `yield from context-manager` syntax when you need to process each relevant entry individually. It allows iterating on complex iterables more cleanly than manually handling these cases.

```python
iterable = [{'id': 1}, {'id': 2}, {'id':3}]
my_iterator = (item['id'] for item in iterable if item['id'] % 4 == 0)
for value in my_iterator:
   print(value)
```
**How to proceed ?**
=====================

*   Start with the problem statement: Identify elements of your data stream you want to work with one at a time. Iterate and handle them individually, using yield where appropriate.

Note: This example code snippet uses Python 3.x features (yield from). Make sure to test in your specific environment before implementation